In [0]:
CREATE OR REFRESH STREAMING TABLE mavenmarket.gold.fact_sales
CLUSTER BY (transaction_date_sk, transaction_mode, store_sk)
COMMENT 'Unified Fact Table containing physical store and online sales transactions'
AS
WITH parsed_transactions AS (
    SELECT 
        transaction_sk AS sales_fact_sk,
        customer_sk,
        product_sk,
        store_sk,
        promotion_sk,
        
        COALESCE(
            to_date(transaction_date, 'MM/dd/yyyy'),
            to_date(transaction_date, 'dd-MMM-yy'),
            to_date(transaction_date, 'yyyy-MM-dd')
        ) AS parsed_transaction_date,

        COALESCE(
            to_date(stock_date, 'MM/dd/yyyy'),
            to_date(stock_date, 'dd-MMM-yy'),
            to_date(stock_date, 'yyyy-MM-dd')
        ) AS parsed_stock_date,

        lower(TRIM(transaction_mode)) AS transaction_mode,
        COALESCE(CAST(quantity AS INT), 0) AS sales_quantity
    FROM STREAM(mavenmarket.silver.silver_transactions)
    WHERE transaction_date IS NOT NULL
)
SELECT 
    t.sales_fact_sk,
    
    -- Date Foreign Keys (mapped directly to dim_date.date_sk)
    COALESCE(
        dt.date_sk, 
        CAST(date_format(t.parsed_transaction_date, 'yyyyMMdd') AS INT)
    ) AS transaction_date_sk,
    
    COALESCE(
        ds.date_sk, 
        CAST(date_format(t.parsed_stock_date, 'yyyyMMdd') AS INT)
    ) AS stock_date_sk,
    
    -- Foreign Keys directly from Silver
    t.product_sk,
    t.customer_sk,
    t.store_sk,
    COALESCE(t.promotion_sk, '0') AS promotion_sk,
    
    LOWER(t.transaction_mode) AS transaction_mode,
    t.sales_quantity,
    
    -- Financial Measures calculated using unit prices from Gold dim_product
    CAST(t.sales_quantity * COALESCE(p.retail_price_amount, 0.00) AS DECIMAL(18,2)) AS gross_revenue_amount,
    CAST(t.sales_quantity * COALESCE(p.unit_cost_amount, 0.00) AS DECIMAL(18,2)) AS total_cost_amount,
    CAST(
        (t.sales_quantity * COALESCE(p.retail_price_amount, 0.00)) - 
        (t.sales_quantity * COALESCE(p.unit_cost_amount, 0.00)) 
        AS DECIMAL(18,2)
    ) AS gross_profit_amount

FROM parsed_transactions t

-- Join with dim_product to fetch unit costs and retail prices
LEFT JOIN mavenmarket.gold.dim_product p 
    ON t.product_sk = p.product_sk

-- Join with dim_date for transaction date surrogate key lookup
LEFT JOIN mavenmarket.gold.dim_date dt 
    ON t.parsed_transaction_date = dt.calendar_date

-- Join with dim_date for stock date surrogate key lookup
LEFT JOIN mavenmarket.gold.dim_date ds 
    ON t.parsed_stock_date = ds.calendar_date;